## ▶ What you'll see when you run this
- A printed **scorecard** comparing a `contains` baseline with an **LLM-as-judge**, plus a **two-judge comparison** (Claude vs Gemini) on the same answers.

**Time:** ~6 min · **Cost:** free (heuristic fallback with no key; a few cents with one) · **Keys:** none | ANTHROPIC_API_KEY and/or GEMINI_API_KEY (optional, for real judges)

# Week 17 · Notebook 1 — A Small Eval Harness (LLM-as-Judge)
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Build a tiny **eval harness**: test cases + scorers + an aggregate report. We score with **`contains`** (a substring baseline) *and* an **LLM-as-judge** (Claude *and* Gemini). We reuse the course's shared **`tools/eval_utils.py`** (`llm_judge`, `scorecard`) — the same helper as Weeks 9, 11, 13 — and show the from-scratch scorers under the hood. **No OpenAI.**

> Degrades gracefully: without an API key the judge falls back to a clear heuristic, and the `contains` scorer still runs offline.

## 0. Install + load keys safely

In [ ]:
!pip -q install anthropic google-generativeai
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    pass  # locally, set these in your shell environment
print('Anthropic key set:', bool(os.environ.get('ANTHROPIC_API_KEY')))
print('Gemini key set:', bool(os.environ.get('GEMINI_API_KEY')))

## 0b. Import the shared eval helpers
The course ships `tools/eval_utils.py` (`llm_judge`, `scorecard`) used across Weeks 9, 11, 13. We search **upward** from the notebook for the top-level `tools/` folder so the import works locally and in Colab. We'll use `scorecard` to print the report card; later we build a judge from scratch too, so you see what's **under the hood** of `llm_judge`.

In [ ]:
import os, sys

def _find_tools_dir(start=None, max_up=6):
    here = os.path.abspath(start or os.getcwd())
    for _ in range(max_up):
        cand = os.path.join(here, 'tools', 'eval_utils.py')
        if os.path.exists(cand):
            return os.path.dirname(cand)
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return None

tools_dir = _find_tools_dir()
if tools_dir:
    sys.path.insert(0, tools_dir)
    from eval_utils import llm_judge, scorecard
    print('Imported eval_utils from:', tools_dir)
else:
    print('eval_utils.py not found nearby — using a minimal inline fallback.')
    def llm_judge(question, answer, rubric='', provider='gemini'):
        score = 3 if (answer and answer.strip()) else 1
        return {'score': score, 'reason': '(inline heuristic fallback)',
                'judge': 'fallback'}
    def scorecard(rows, score_key='score'):
        n = len(rows) or 1
        avg = sum(float(r.get(score_key, 0) or 0) for r in rows) / n
        print('SCORECARD', len(rows), 'cases  | avg', round(avg, 2))
        return {'n': len(rows), 'avg': avg}

## 1. The system under test
Any function `input -> output` works. We use Claude, but you would plug in your Final Project pipeline (RAG, agent, fine-tuned model, ...).

In [ ]:
MODEL = 'claude-haiku-4-5-20251001'

def system_under_test(question: str) -> str:
    try:
        import anthropic
        client = anthropic.Anthropic()
        msg = client.messages.create(
            model=MODEL, max_tokens=100,
            messages=[{'role': 'user',
                       'content': f'Answer briefly: {question}'}])
        return msg.content[0].text.strip()
    except Exception as e:
        return f'[offline/no key: {e}]'

print(system_under_test('What is the capital of France?'))

## 2. Test cases
Each case has an input and a reference of what 'good' looks like. 5-10 cases is enough to start; grow them as you find failures.

In [ ]:
test_cases = [
    {'input': 'What is the capital of France?', 'expected': 'Paris'},
    {'input': 'What is 2 + 2?',                 'expected': '4'},
    {'input': 'What language is this course taught in?', 'expected': 'Python'},
    {'input': 'Who wrote Romeo and Juliet?',    'expected': 'Shakespeare'},
]
print(len(test_cases), 'test cases')

## 3. Scorer A — `contains` (substring match)
Objective and free. This is **substring containment**, *not* strict exact-match — we call it `contains` to be precise (strict exact-match would compare the whole normalized strings; `eval_utils.exact_match` does that). Good for single-answer tasks; brittle for free-form text ('Paris.' vs 'The capital is Paris').

In [ ]:
def contains_score(answer: str, expected: str) -> float:
    return 1.0 if expected.lower() in answer.lower() else 0.0

print(contains_score('The capital is Paris.', 'Paris'))  # 1.0
print(contains_score('I am not sure.', 'Paris'))         # 0.0

## 4. Scorer B — LLM-as-judge (Claude)
A strong model grades against a rubric. We ask for a single number and keep temperature low. **Judge hygiene:** clear rubric, structured output, spot-check against your own judgment, and counter the three documented **judge biases**: **position bias** (favoring whichever answer is shown first — swap order and average), **verbosity bias** (over-rewarding longer answers — score conciseness), and **self-preference bias** (rating its own model family higher — use a cross-family judge, which we do in §6).

In [ ]:
def judge_score(answer: str, expected: str, question: str = '') -> float:
    prompt = (
        'You are a strict grader. Score the model answer for correctness.\n'
        f'Reference answer: {expected}\n'
        f'Model answer: {answer}\n'
        'Reply with ONLY 1 (correct) or 0 (incorrect).')
    try:
        import anthropic
        client = anthropic.Anthropic()
        out = client.messages.create(
            model='claude-sonnet-4-6', max_tokens=5, temperature=0,
            messages=[{'role': 'user', 'content': prompt}]).content[0].text
        return 1.0 if '1' in out else 0.0
    except Exception as e:
        print('   (judge offline:', e, ')')
        return float('nan')

print('judge demo:', judge_score('It is Paris.', 'Paris'))

### Gemini judge alternative
Same idea with Gemini — useful for a second opinion or if you only have a Gemini key.

In [ ]:
def gemini_judge(answer: str, expected: str) -> float:
    try:
        import google.generativeai as genai
        genai.configure(api_key=os.environ['GEMINI_API_KEY'])
        model = genai.GenerativeModel('gemini-2.5-flash')
        prompt = (f'Reference: {expected}\nAnswer: {answer}\n'
                  'Reply ONLY 1 if the answer is correct, else 0.')
        out = model.generate_content(
            prompt,
            generation_config={'temperature': 0, 'max_output_tokens': 5}).text
        return 1.0 if '1' in out else 0.0
    except Exception as e:
        print('   (gemini judge offline:', e, ')')
        return float('nan')

## 5. The harness — run, score, aggregate
This is the whole point: one function that runs every case, scores it, and returns an average plus a per-case table.

In [ ]:
def run_eval(system_fn, score_fn):
    rows, scores = [], []
    for tc in test_cases:
        answer = system_fn(tc['input'])
        s = score_fn(answer, tc['expected'])
        rows.append((tc['input'], answer, tc['expected'], s))
        if s == s:  # skip NaN (judge offline)
            scores.append(s)
    avg = sum(scores) / len(scores) if scores else float('nan')
    return avg, rows

avg, rows = run_eval(system_under_test, contains_score)
print(f'\ncontains average: {avg:.2f}\n')
for q, a, exp, s in rows:
    print(f'  [{s}] Q: {q}\n       A: {a}\n       expected ~ {exp}\n')

# Same data through the shared scorecard helper from eval_utils.
scorecard([{'q': q, 'score': s} for q, a, exp, s in rows])

## 6. Run TWO LLM judges and compare (watch for self-preference bias)
Re-run with **both** judges — Claude (`claude-sonnet-4-6`) and Gemini (`gemini-2.5-flash`) — on the same answers and compare their averages. The system under test is `claude-haiku-4-5-20251001`, so the Claude judge is **same-family**: LLM judges show a measurable **self-preference bias**, tending to rate answers from their own model family more favorably, so a **cross-family** judge (here, Gemini) or human spot-checks are a useful sanity check. Both judges are wired up (no dead code).

In [ ]:
claude_avg, _ = run_eval(system_under_test, lambda a, e: judge_score(a, e))
gemini_avg, _ = run_eval(system_under_test, lambda a, e: gemini_judge(a, e))
print(f'Claude judge (same family as SUT): {claude_avg:.2f}')
print(f'Gemini judge (cross family)      : {gemini_avg:.2f}')
if claude_avg == claude_avg and gemini_avg == gemini_avg:
    print(f'gap (claude - gemini): {claude_avg - gemini_avg:+.2f}  '
          '(a positive gap can hint at same-family self-preference)')
else:
    print('(set ANTHROPIC_API_KEY and GEMINI_API_KEY to compare real judges)')
judge_avg = claude_avg  # keep the Claude judge for the record below

## 7. Track it over time
Save the score with a label (model + prompt version). Re-run after every change to prove an improvement — and to justify a cheaper model.

In [ ]:
import datetime
record = {'when': datetime.datetime.now().isoformat(timespec='minutes'),
          'model': MODEL, 'contains': round(avg, 3),
          'judge': None if judge_avg != judge_avg else round(judge_avg, 3)}
print('eval record:', record)
# Append records to a CSV/JSON in your repo to chart quality over versions.

## 8. Use this in your Final Project
- Swap `system_under_test` for **your** pipeline.
- Add 5-10 test cases that matter for your problem.
- Report the average score in your write-up.
- Pair this with the **safety + Responsible AI checklist** (hallucination, bias & fairness, prompt injection, privacy/PII, transparency/disclosure, societal impact) from this week's document.